In [1]:
from WKAN import *

CUDA is available. Using GPU.


In [2]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from kan import KAN as OriginalKAN

# Set seed for reproducibility
torch.manual_seed(114514)

class EfficientKAN2Layer(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, grid_size, spline_order):
        super().__init__()
        self.layer1 = KANLinear(in_features, hidden_features, grid_size=grid_size, spline_order=spline_order)
        self.layer2 = KANLinear(hidden_features, out_features, grid_size=grid_size, spline_order=spline_order)
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

class EfficientKAN3Layer(nn.Module):
    def __init__(self, in_features, hidden1, hidden2, out_features, grid_size, spline_order):
        super().__init__()
        self.layer1 = KANLinear(in_features, hidden1, grid_size=grid_size, spline_order=spline_order)
        self.layer2 = KANLinear(hidden1, hidden2, grid_size=grid_size, spline_order=spline_order)
        self.layer3 = KANLinear(hidden2, out_features, grid_size=grid_size, spline_order=spline_order)
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

# Train the model
def train(model, x, y, epochs=100):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    start_time = time.time()
    for epoch in range(epochs):
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
    return time.time() - start_time

# Experiment 1: Simple Regression
x1 = torch.linspace(-1, 1, 100).unsqueeze(1)
y1 = torch.sin(3.14 * x1)
original_model1 = OriginalKAN(width=[1, 5, 1], grid=5, k=3)
efficient_model1 = EfficientKAN2Layer(1, 5, 1, 5, 3)
original_time1 = train(original_model1, x1, y1)
efficient_time1 = train(efficient_model1, x1, y1)
print(f"Experiment 1 - Original KAN Time: {original_time1:.4f}s, Efficient KAN Time: {efficient_time1:.4f}s")

# Experiment 2: Medium Regression
x2 = torch.rand(1000, 2) * 2 - 1
y2 = x2[:, 0]**2 + x2[:, 1]**2
original_model2 = OriginalKAN(width=[2, 10, 1], grid=5, k=3)
efficient_model2 = EfficientKAN2Layer(2, 10, 1, 5, 3)
original_time2 = train(original_model2, x2, y2.unsqueeze(1))
efficient_time2 = train(efficient_model2, x2, y2.unsqueeze(1))
print(f"Experiment 2 - Original KAN Time: {original_time2:.4f}s, Efficient KAN Time: {efficient_time2:.4f}s")

# Experiment 3: Larger Regression
x3 = torch.rand(10000, 5) * 2 - 1
y3 = (x3**2).sum(dim=1, keepdim=True)
original_model3 = OriginalKAN(width=[5, 10, 10, 1], grid=5, k=3)
efficient_model3 = EfficientKAN3Layer(5, 10, 10, 1, 5, 3)
original_time3 = train(original_model3, x3, y3)
efficient_time3 = train(efficient_model3, x3, y3)
print(f"Experiment 3 - Original KAN Time: {original_time3:.4f}s, Efficient KAN Time: {efficient_time3:.4f}s")

# Experiment 4: Complex Model
x4 = x3  # Reuse dataset from Experiment 3
original_model4 = OriginalKAN(width=[5, 10, 10, 1], grid=10, k=5)
efficient_model4 = EfficientKAN3Layer(5, 10, 10, 1, 10, 5)
original_time4 = train(original_model4, x4, y3)
efficient_time4 = train(efficient_model4, x4, y3)
print(f"Experiment 4 - Original KAN Time: {original_time4:.4f}s, Efficient KAN Time: {efficient_time4:.4f}s")

Experiment 1 - Original KAN Time: 1.1424s, Efficient KAN Time: 0.4153s
Experiment 2 - Original KAN Time: 2.7617s, Efficient KAN Time: 1.2215s
Experiment 3 - Original KAN Time: 53.9806s, Efficient KAN Time: 5.6666s
Experiment 4 - Original KAN Time: 160.3346s, Efficient KAN Time: 15.6581s
